In [ ]:
# -----------------------------
# 0. Setup & Imports
# -----------------------------
import os
from typing import List
from pydantic import BaseModel
from dotenv import load_dotenv
import gradio as gr

from langchain_core.documents import Document   # updated import
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq
from langchain_community.embeddings import HuggingFaceEmbeddings   # Hugging Face embeddings
from langgraph.graph import StateGraph, END


In [5]:
# Load environment variables
load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [6]:
# -----------------------------
# 1. Prepare Vectorstore
# -----------------------------
docs = TextLoader(
    "C:/Users/admin/Desktop/New_GenAI/GenAI/LangGraph/Autonomus RAG/xumo_manual_rag.txt",
    encoding="utf-8"
).load()

In [7]:
# Create a text splitter that breaks the manual into chunks of 500 characters
# with an overlap of 50 characters to preserve context between chunks
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)

# Apply the splitter to the loaded documents so we get a list of smaller chunks
chunks = splitter.split_documents(docs)

In [8]:
# Hugging Face embeddings
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever()

In [9]:
# -----------------------------
# 2. Initialize Groq LLM
# -----------------------------
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    api_key=os.getenv("GROQ_API_KEY")
)

In [11]:
# -----------------------------
# 3. LangGraph State Definition
# -----------------------------
class ManualRAGState(BaseModel):   
    question: str
    sub_steps: List[str] = []
    retrieved_docs: List[Document] = []
    answer: str = ""

In [12]:
# -----------------------------
# 4. Nodes
# -----------------------------
def plan_steps(state: ManualRAGState) -> ManualRAGState:
    prompt = f"Break the question into 2-3 reasoning steps:\n\n{state.question}"
    result = llm.invoke(prompt).content
    subqs = [line.strip("- ") for line in result.split("\n") if line.strip()]
    return state.model_copy(update={"sub_steps": subqs})

def retrieve_per_step(state: ManualRAGState) -> ManualRAGState:
    all_docs = []
    for sub in state.sub_steps:
        docs = retriever.invoke(sub)
        all_docs.extend(docs)
    return state.model_copy(update={"retrieved_docs": all_docs})

def generate_answer(state: ManualRAGState) -> ManualRAGState:
    context = "\n\n".join([doc.page_content for doc in state.retrieved_docs])
    prompt = f"""
You are answering a complex question using reasoning and retrieved documents.

Question: {state.question}

Relevant Information:
{context}

Now synthesize a well-reasoned final answer.
"""
    result = llm.invoke(prompt).content.strip()
    return state.model_copy(update={"answer": result})


In [13]:
# -----------------------------
# 5. LangGraph Graph
# -----------------------------
builder = StateGraph(ManualRAGState)
builder.add_node("planner", plan_steps)
builder.add_node("retriever", retrieve_per_step)
builder.add_node("responder", generate_answer)

builder.set_entry_point("planner")
builder.add_edge("planner", "retriever")
builder.add_edge("retriever", "responder")
builder.add_edge("responder", END)

graph = builder.compile()


In [15]:
# -----------------------------
# 6. Gradio Interface
# -----------------------------
def rag_pipeline(user_query: str):
    state = ManualRAGState(question=user_query)
    final = graph.invoke(state)
    steps = "\n".join(final["sub_steps"])
    return f"Reasoning Steps:\n{steps}\n\nFinal Answer:\n{final['answer']}"

demo = gr.Interface(
    fn=rag_pipeline,
    inputs=gr.Textbox(label="Ask about Xumo Stream Box Manual"),
    outputs=gr.Textbox(label="RAG Answer"),
    title="Xumo Manual RAG Assistant"
)

if __name__ == "__main__":
    demo.launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
